In [51]:
import langchain

In [52]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [54]:
#example 1: simple llm call with streaming
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage



In [55]:
model = init_chat_model("groq:llama-3.1-8b-instant")
model


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000000CC2A98EFD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000000CC2A98F9D0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [56]:
messages = [
    SystemMessage("you are an helpful ai assistant"),
    HumanMessage("what are the two benifits of using langchain")
]

In [57]:
##invoke the model
response = model.invoke(messages)
response

AIMessage(content='LangChain is an open-source library that enables users to build and interact with large language models (LLMs) in a more intuitive and efficient way. Here are two benefits of using LangChain:\n\n1. **Improved Interactivity**: LangChain allows users to interact with LLMs in a more natural and conversational way. It provides a set of APIs and tools that enable users to generate, retrieve, and manipulate text data, making it easier to build chatbots, voice assistants, and other conversational AI systems. This interactivity enables users to ask follow-up questions, provide context, and receive more accurate and relevant responses from the LLM.\n\n2. **Increased Efficiency**: LangChain streamlines the process of working with LLMs by providing a unified API and a set of pre-built functions that simplify common tasks, such as data preparation, model loading, and response generation. This reduces the time and effort required to build and deploy AI-powered applications, makin

In [58]:
## streaming example
for chunk in model.stream(messages):
    print(chunk.content,end="",flush=True)

LangChain is an open-source framework for building conversational AI models. Two benefits of using LangChain are:

1. **Modularity and Flexibility**: LangChain allows developers to build conversational AI models by combining different components, such as natural language processing (NLP) tools, data storage, and machine learning algorithms, into a single framework. This modularity and flexibility enable developers to easily swap out or add new components as needed, making it easier to adapt to changing requirements and new technologies.

2. **Streamlined Development and Deployment**: LangChain provides a set of pre-built tools and libraries that simplify the process of developing and deploying conversational AI models. This includes features such as data loading, model fine-tuning, and deployment to cloud platforms. By using LangChain, developers can accelerate their development process, reduce the complexity of building conversational AI models, and focus on higher-level tasks such as

In [59]:
### dynamic prompt templates
from langchain_core.prompts import ChatPromptTemplate

###create translation app

translation_template = ChatPromptTemplate.from_messages([
        ("system", "You are a professional translator.Translate the following text {text} from {source_language} to  {target_language}. Maintain the tone and style"),
        ("user", "{text}")
    ])

prompt = translation_template.invoke({
        "source_language": "English",
        "target_language": "Spanish",
        "text": "langchain making building AI application incredibiy easy"
    }
)

In [60]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator.Translate the following text langchain making building AI application incredibiy easy from English to  Spanish. Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='langchain making building AI application incredibiy easy', additional_kwargs={}, response_metadata={})])

In [61]:
translated_response = model.invoke(prompt)
print(translated_response.content)

Langchain: Haciendo que construir aplicaciones de inteligencia artificial sea increíblemente fácil.

Nota: Langchain es un término que se utiliza para describir una plataforma de desarrollo de inteligencia artificial que permite a los desarrolladores crear aplicaciones de IA de manera sencilla y rápida.

Traducción literal: Langchain es una herramienta que facilita enormemente el proceso de construcción de aplicaciones de inteligencia artificial.

Sin embargo, para mantener el tono y estilo originales, una posible traducción más fluida podría ser:

Langchain: La revolución en la creación de aplicaciones de inteligencia artificial, ¡hacía que sea increíblemente fácil!


In [62]:
###Buiding your first chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

def create_story_chain():
    story_prompt = ChatPromptTemplate.from_messages(
        [
            ("system","you are an creative storyteller. Write a short and engagin story BASED ON A GIVEN THEME, character and settings"),
            ("user", "Theme {theme}\n Main Character: {character} \n Setting: {setting}")
        ]
    )

    #Template for story analysis
    analysis_prompt = ChatPromptTemplate.from_messages(
        [
            ("system","you are an literary critic. Analyze the following story and provide insights"),
            ("user", "{story}")
        ]
    ) 


    story_chain=(
        story_prompt | model | StrOutputParser
    )

    def analyze_story(story_text):
        return {"story": story_text}

    analysis_chain = (
        story_chain
        | RunnableLambda(analyze_story)
        | analysis_prompt
        | model
        | StrOutputParser()
    )

    return analysis_chain


In [ ]:
chain=create_story_chain()
result=chain.invoke(
    {
      "theme": "artificial intelligence",
      "character": "a curious robot",
      "setting": "a futuristic story"
    })

KeyError: "Input to ChatPromptTemplate is missing variables {'setting', 'character'}.  Expected: ['character', 'setting', 'theme'] Received: ['theme']\nNote: if you intended {setting} to be part of the string and not a variable, please escape it with double curly braces like: '{{setting}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "